In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_J.xpt"
csv_file = DATA_DIR / "WHQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
# WHQ_J requires NO validated numeric correction.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 WHQ_J XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHD080T",
    "WHD080U",
    "WHD080L",
    "WHQ225",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150",
    "WHQ190",
    "WHQ200"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "WHQ_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6161:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 37:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,161 rows x 37 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of the attached WHQ_J XPT showed:
#
#     0 genuine fractional observations
#
# across all 37 fields.
#
# Therefore every variable in THIS PARTICULAR FILE
# is a whole-number value/code field.
#
# This includes self-reported height and weight variables,
# because the released WHQ_J values themselves are stored
# as whole-number values.
#
# IMPORTANT:
#
# This applies ONLY to WHQ_J.
#
# Do NOT use this assumption for:
#
#     Dietary
#     Laboratory
#     Examination
#     survey weights
#     nutrients
#     environmental exposures
#     other genuine decimal variables
#
# Those decimals must remain untouched.
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT/pandas decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER replace this globally without first validating
# the variable/count against CDC.
#
# Attached WHQ_J:
#
#     tiny artifacts = 0
#
# Therefore NO tiny-value correction is permitted.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 37 WHQ_J fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT TINY-VALUE CHECK ---")

print(
    "Tiny XPORT artifacts:",
    tiny_total
)


if tiny_total > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in WHQ_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: WHQ_J contains no tiny XPORT artifacts."
)


# ============================================================
# 6. CHECK ORIGINAL NUMERIC ZERO PATTERN
# ============================================================

#
# Direct inspection of attached WHQ_J:
#
#     ordinary numeric zeros = 0
#
# Therefore no zero restoration is required.
#

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL NUMERIC ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if true_zero_total > 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "found in WHQ_J."
    )


print(
    "PASS: Original WHQ_J contains no numeric zeros."
)


# ============================================================
# 7. EXPECTED CORRECTED SOURCE
# ============================================================

#
# WHQ_J needs NO numeric repair.
#
# Therefore the expected corrected values are
# exactly the original XPT-derived values.
#

expected_df = source_df.copy()


# ============================================================
# 8. CHECK EVERY COLUMN FOR GENUINE FRACTIONS
# ============================================================

#
# CRITICAL:
#
# This check occurs BEFORE any Int64 conversion.
#
# np.round() below is ONLY used to TEST whether values
# are integral.
#
# It does NOT modify df.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in WHQ_J.\n\n"
        "They will NOT be rounded or converted to Int64:\n"
        + str(fractional_columns)
    )


print(
    "\nPASS: No genuine fractional values "
    "in any of the 37 WHQ_J fields."
)


# ============================================================
# 9. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 10. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 37 variables were tested ABOVE.
#
# Therefore Int64 conversion is safe for this file.
#
# Examples:
#
#     93705.0 -> 93705
#     170.0   -> 170
#     1.0     -> 1
#
# NO .round() transformation is used.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 37 WHQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. EXACT FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. RANGE-SUMMARY CHECK HELPER
# ============================================================

def check_range_summary(
    column,
    lower,
    upper,
    expected_range_count,
    special_counts,
    expected_missing
):

    values = df[column]

    range_count = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )

    actual_missing = int(
        values.isna()
        .sum()
    )

    print(
        f"\n--- CDC RANGE CHECK: {column} ---"
    )

    print(
        f"Range {lower}-{upper}:",
        f"{range_count:,}"
    )

    if range_count != expected_range_count:

        raise ValueError(
            f"{column} range count differs from CDC.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )

    for code, expected_count in special_counts.items():

        actual_count = int(
            values.eq(code)
            .sum()
        )

        print(
            f"Code {code}:",
            f"{actual_count:,}"
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} differs from CDC.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} CDC range summary matches."
    )


# ============================================================
# 13. CDC CHECK - WHD010
# ============================================================

#
# Current self-reported height, inches.
#

check_range_summary(
    "WHD010",
    49,
    82,
    5977,
    {
        7777: 1,
        9999: 128
    },
    55
)


# ============================================================
# 14. CDC CHECK - WHD020
# ============================================================

#
# Current self-reported weight, pounds.
#

check_range_summary(
    "WHD020",
    72,
    484,
    6048,
    {
        7777: 9,
        9999: 104
    },
    0
)


# ============================================================
# 15. CDC CHECK - WHQ030
# ============================================================

check_frequency(
    "WHQ030",
    {
        1: 3060,
        2: 376,
        3: 2708,
        9: 17
    },
    0
)


# ============================================================
# 16. CDC CHECK - WHQ040
# ============================================================

check_frequency(
    "WHQ040",
    {
        1: 592,
        2: 3752,
        3: 1812,
        9: 5
    },
    0
)


# ============================================================
# 17. CDC CHECK - WHD050
# ============================================================

#
# Self-reported weight one year ago.
#

check_range_summary(
    "WHD050",
    67,
    520,
    6052,
    {
        7777: 4,
        9999: 105
    },
    0
)


# ============================================================
# 18. CDC CHECK - WHQ060
# ============================================================

check_frequency(
    "WHQ060",
    {
        1: 778,
        2: 495
    },
    4888
)


# ============================================================
# 19. CDC CHECK - WHQ070
# ============================================================

check_frequency(
    "WHQ070",
    {
        1: 2122,
        2: 3254
    },
    785
)


# ============================================================
# 20. WEIGHT-LOSS METHOD VARIABLES
# ============================================================

#
# These are "code all that apply" variables.
#
# Each variable contains only its documented method
# code when selected; otherwise it is missing.
#

METHOD_CHECKS = {

    "WHD080A": (
        {10: 1752},
        4409
    ),

    "WHD080B": (
        {11: 955},
        5206
    ),

    "WHD080C": (
        {12: 957},
        5204
    ),

    "WHD080D": (
        {13: 1808},
        4353
    ),

    "WHD080E": (
        {14: 630},
        5531
    ),

    "WHD080F": (
        {15: 250},
        5911
    ),

    "WHD080G": (
        {16: 192},
        5969
    ),

    "WHD080H": (
        {17: 109},
        6052
    ),

    "WHD080I": (
        {31: 84},
        6077
    ),

    "WHD080J": (
        {32: 145},
        6016
    ),

    "WHD080K": (
        {33: 39},
        6122
    ),

    "WHD080M": (
        {34: 1631},
        4530
    ),

    "WHD080N": (
        {30: 188},
        5973
    ),

    "WHD080O": (
        {41: 919},
        5242
    ),

    "WHD080P": (
        {42: 35},
        6126
    ),

    "WHD080Q": (
        {43: 1515},
        4646
    ),

    "WHD080R": (
        {44: 1146},
        5015
    ),

    "WHD080S": (
        {45: 1237},
        4924
    ),

    "WHD080T": (
        {46: 1339},
        4822
    ),

    "WHD080U": (
        {35: 17},
        6144
    ),

    "WHD080L": (
        {40: 43},
        6118
    )
}


for column, (
    expected_values,
    expected_missing
) in METHOD_CHECKS.items():

    check_frequency(
        column,
        expected_values,
        expected_missing
    )


print(
    "\nPASS: All WHD080 weight-loss method "
    "frequencies match CDC."
)


# ============================================================
# 21. CDC CHECK - WHQ225
# ============================================================

check_frequency(
    "WHQ225",
    {
        1: 1659,
        2: 872,
        3: 291,
        4: 287,
        5: 3015,
        9: 37
    },
    0
)


# ============================================================
# 22. CDC CHECK - WHD110
# ============================================================

#
# Self-reported weight 10 years ago.
#

check_range_summary(
    "WHD110",
    72,
    630,
    4093,
    {
        7777: 8,
        9999: 117
    },
    1943
)


# ============================================================
# 23. CDC CHECK - WHD120
# ============================================================

#
# Self-reported weight at age 25.
#

check_range_summary(
    "WHD120",
    70,
    530,
    4764,
    {
        7777: 4,
        9999: 230
    },
    1163
)


# ============================================================
# 24. CDC CHECK - WHD130
# ============================================================

#
# Self-reported height at age 25.
#

check_range_summary(
    "WHD130",
    41,
    89,
    2915,
    {
        7777: 1,
        9999: 122
    },
    3123
)


# ============================================================
# 25. CDC CHECK - WHD140
# ============================================================

#
# Greatest self-reported weight.
#

check_range_summary(
    "WHD140",
    90,
    630,
    5773,
    {
        7777: 6,
        9999: 76
    },
    306
)


# ============================================================
# 26. CDC CHECK - WHQ150
# ============================================================

#
# Age when heaviest.
#
# CDC top-codes age 80+ as 80.
#

check_range_summary(
    "WHQ150",
    10,
    79,
    5636,
    {
        80: 91,
        77777: 1,
        99999: 42
    },
    391
)


# ============================================================
# 27. CDC CHECK - WHQ190
# ============================================================

check_frequency(
    "WHQ190",
    {
        1: 74,
        2: 6087
    },
    0
)


# ============================================================
# 28. CDC CHECK - WHQ200
# ============================================================

#
# Age at most recent weight-loss surgery.
#

check_range_summary(
    "WHQ200",
    21,
    79,
    73,
    {
        77777: 0,
        99999: 1
    },
    6087
)


# ============================================================
# 29. WHQ190 -> WHQ200 ROUTING CHECK
# ============================================================

#
# WHQ200 is populated exactly for respondents
# reporting weight-loss surgery:
#
#     WHQ190 = 1
#
# Attached file:
#
#     WHQ190=1       = 74
#     WHQ200 present = 74
#

surgery_yes = (
    df["WHQ190"]
    .eq(1)
    .to_numpy(
        dtype=bool
    )
)


surgery_age_present = (
    df["WHQ200"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


surgery_route_mismatches = int(
    np.sum(
        surgery_yes
        !=
        surgery_age_present
    )
)


print(
    "\n--- WHQ190 / WHQ200 ROUTING ---"
)

print(
    "WHQ190 = Yes:",
    int(surgery_yes.sum())
)

print(
    "WHQ200 populated:",
    int(surgery_age_present.sum())
)

print(
    "Routing mismatches:",
    surgery_route_mismatches
)


if surgery_route_mismatches != 0:

    raise ValueError(
        "WHQ200 routing does not exactly "
        "match WHQ190=1."
    )


print(
    "PASS: All 74 weight-loss surgery records "
    "route exactly to WHQ200."
)


# ============================================================
# 30. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

#
# Original WHQ_J contained:
#
#     tiny zero artifacts = 0
#     ordinary zeros      = 0
#
# No values were repaired.
#
# Therefore the final zero pattern must remain empty.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zero values introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zero values introduced."
)


# ============================================================
# 31. COMPLETE VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Permitted numeric modifications in WHQ_J:
#
#     NONE
#
# Therefore every current value must exactly match
# the original XPT-derived value.
#

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 37 variables exactly match "
    "their original XPT numeric values."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "before CSV export."
    )


print(
    "PASS: Missing positions unchanged."
)


# ============================================================
# 33. EXPORT CSV
# ============================================================

#
# FIDELITY POLICY
# ------------------------------------------------------------
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     blanket decimal formatting
#
# This particular WHQ_J file has no genuine decimal
# variables, but we retain the same export philosophy
# used throughout the NHANES pipeline.
#
# Genuine Dietary / Lab / Exam decimal variables in
# other files should remain AS IS rather than being
# rounded simply to remove ".0" elsewhere.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 34. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 35. EXACT ORIGINAL XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 37 CSV variables round-trip "
    "to exact original XPT numeric values."
)


# ============================================================
# 36. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- CSV MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 37. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    6161,
    37
):

    raise ValueError(
        "Raw-text CSV structure changed."
    )


# ============================================================
# 38. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts "
        "found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 39. RAW CSV .0 CHECK
# ============================================================

#
# WHQ_J has been proven to contain no genuine decimal
# variables.
#
# Therefore it is safe to inspect all 37 variables
# for unnecessary trailing ".0".
#
# Examples:
#
#     93705.0 -> 93705
#     170.0   -> 170
#     1.0     -> 1
#
# IMPORTANT:
#
# Do NOT apply this all-column check to a mixed file
# containing genuine decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any WHQ_J field."
)


# ============================================================
# 40. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN 93705-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL COMPLETE SOURCE -> CSV FIDELITY
# ============================================================

#
# Final all-variable numeric comparison.
#
# WHQ_J permitted numeric corrections:
#
#     NONE
#

final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    original_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    final_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    same = np.allclose(
        original_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves exact original "
    "numeric values for all 37 variables."
)


# ============================================================
# 44. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "WHQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - Weight History"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Original ordinary numeric zeros:",
    true_zero_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Other numeric corrections:",
    0
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "WHQ190 weight-loss surgery = Yes:",
    int(
        df["WHQ190"]
        .eq(1)
        .sum()
    )
)

print(
    "WHQ200 age records:",
    int(
        df["WHQ200"]
        .notna()
        .sum()
    )
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 WHQ_J XPT -> CSV
Rows: 6,161
Columns: 37
PASS: Structure = 6,161 rows x 37 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: 37

--- ORIGINAL XPT TINY-VALUE CHECK ---
Tiny XPORT artifacts: 0
PASS: WHQ_J contains no tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
Ordinary numeric zeros: 0
PASS: Original WHQ_J contains no numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
WHD010: 0 fractional observations
WHD020: 0 fractional observations
WHQ030: 0 fractional observations
WHQ040: 0 fractional observations
WHD050: 0 fractional observations
WHQ060: 0 fractional observations
WHQ070: 0 fractional observations
WHD080A: 0 fractional observations
WHD080B: 0 fractional observations
WHD080C: 0 fractional observations
WHD080D: 0 fractional observations
WHD080E: 0 fractional observations
WHD080F: 0 fractional observations
WHD080G: 0 fractional observations
WHD080H: 0 fractional observations
WHD080I: 0